In [14]:
import matplotlib.pyplot as plt
import sys
from typing import List, Dict
if not '../..' in sys.path:
    sys.path.append('../..')
import basico
import numpy
import pandas
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import random
import itertools
import math
import tqdm
import libsbml
import pickle
import time
import os

In [ ]:
fname = "../Simulation_data.xlsx"
data = pandas.read_excel(fname, sheet_name=1)


In [16]:
data

,time,adp,asa,asp,aspp,atp,hs,hsp,nadp,nadph,phos,thr,Experiment
0,0,0.000000,0.000000,1.292900,0.000000,6.464400,0.000000,0.000000,0.000000,2.000000,0.000000,2.000000,1
1,1,0.000000,0.013994,1.123084,0.003921,4.333367,0.000000,0.018433,0.000000,1.836330,0.000000,1.747953,1
2,2,0.000000,0.021633,1.984576,0.004058,9.364119,0.004227,0.013632,0.000000,1.878701,0.000000,1.758829,1
3,3,0.013249,0.027716,0.592398,0.003463,6.618906,0.000000,0.006901,0.484441,1.879578,0.000000,1.947638,1
4,4,0.000000,0.046001,1.100811,0.004420,5.978385,0.076172,0.033020,0.031116,1.765170,0.415744,1.651898,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
904,96,2.263438,0.009160,1.140171,0.003236,5.644891,0.026977,0.024238,1.472780,0.436031,2.321581,3.015630,9
905,97,2.310092,0.004245,0.999620,0.001421,10.916923,0.018037,0.000000,1.613465,0.092504,2.098442,2.570191,9
906,98,2.509200,0.003017,1.258741,0.002973,7.784833,0.002394,0.028848,1.855979,0.022572,2.544461,2.774298,9
907,99,1.860823,0.007597,1.402961,0.002020,8.551870,0.002968,0.011389,1.798756,0.204134,2.269433,2.993754,9


In [17]:
print(data.columns)

Index(['time', 'adp', 'asa', 'asp', 'aspp', 'atp', 'hs', 'hsp', 'nadp',
       'nadph', 'phos', 'thr', 'Experiment'],
      dtype='object')


In [ ]:
with open("../chass_testbatch.txt", "r") as f:
    testbatch_str = f.read().split()
    testbatch = set([int(el) for el in testbatch_str])
prev_val = 0
batches_lengths = []
for val in data["Experiment"]:
    if val != prev_val:
        batches_lengths.append(1)
        prev_val = val
    else:
        batches_lengths[-1] += 1
not_testbatch = set(range(len(batches_lengths))) - testbatch
not_testbatch_exp_labels = [el + 1 for el in not_testbatch]

In [19]:
maxes = {}
mins = {}
concentrations_labels = ['adp', 'asa', 'asp', 'aspp', 'atp', 'hs', 'hsp', 'nadp', 'nadph', 'phos', 'thr']
for label in (['time'] + concentrations_labels):
    max_val = max(data[data["Experiment"].isin(not_testbatch_exp_labels)][label])
    min_val = min(data[data["Experiment"].isin(not_testbatch_exp_labels)][label])
    label_scaled = f"{label}_scaled"
    maxes[label] = max_val
    mins[label] = min_val
    data[label_scaled] = (data[label] - min_val) / (max_val - min_val)

In [20]:
batches = []
for i, length in enumerate(batches_lengths):
    batches.append(data.iloc[i*length : i*length + length].reset_index(drop=True))
length = batches_lengths[0]

#scale inputs
changing_vars = ['asp', 'atp']
first_of_each_batch = data[changing_vars].iloc[::length]
mins_Xs = first_of_each_batch.min()
maxes_Xs = first_of_each_batch.max()
scaled_initial_conditions = (first_of_each_batch - mins_Xs)/(maxes_Xs-mins_Xs)

Xs_ys = []
for i, batch in enumerate(batches):
    scaled_initials_for_this_batch = pandas.concat([scaled_initial_conditions.iloc[i:i+1]]*length, ignore_index=True)
    X = pandas.concat([batch[["time_scaled"]], scaled_initials_for_this_batch], axis=1)
    y = batch[[f"{label}_scaled" for label in concentrations_labels]]
    X = torch.Tensor(X.values)
    y = torch.Tensor(y.values)
    Xs_ys.append((X, y))

In [ ]:
DROPOUT = 0
INPUTS = 3
OUTPUTS = 11
STOCH_REG_BATCH_SIZE = 16
LR = 0.0001

class ChassFfnnTanhRegressor(nn.Module):
    def __init__(self, inputs, layers, outputs):
        super().__init__()
        all_layers = [inputs] + layers + [outputs]
        self.linears = nn.ModuleList([nn.Linear(all_layers[i], all_layers[i + 1]) for i in range(len(all_layers) - 1)])
        self.act = nn.Tanh()
        self.dropout = nn.Dropout(DROPOUT)
    
    def forward(self, x):
        for hidden in self.linears[:-1]:
            x = self.act(hidden(x))
            x = self.dropout(x)
        x = self.linears[-1](x)
        return x
 

loss_fn = nn.MSELoss()



In [ ]:
n_epochs = 1000000

num_batches = len(batches_lengths)
with open("../chass_train_validation_test", "rb") as file:
    train_validation_test = pickle.load(file)
nums_nodes_layers = [[5, 5], [10, 10], [16, 16], [16, 16, 16]]
num_repetitions = 3
time_to_wait_for_improvement = 2000

In [23]:
num_train_batches = len(train_validation_test[0][0])
example_y = Xs_ys[0][1]
n_train = example_y.numel() * num_train_batches
print(f"Number of data points is {n_train}")

Number of data points is 5555


In [24]:
def AICc(nw, model, n_train, dataloader):
    for X, y in dataloader:
        y_pred = model(X)
        Train_Loss = loss_fn(y_pred, y)
        aic = n_train * math.log(Train_Loss) + 2*nw + 2*nw*(nw+1)/(n_train-nw-1)
        return aic

In [25]:
counter = 0
while os.path.exists(f"chass_ffnn_tanh_{counter}.csv"):
    counter += 1
csv_name = f"chass_ffnn_tanh_{counter}.csv"

In [ ]:
newpath = "saved_models"
if not os.path.exists(newpath):
    os.makedirs(newpath)

In [ ]:

csv_counter = 0
csv_max = 100
csv_first = True
columns = ['num_nodes_on_layer', 'start_weights_iteration', 'k_fold', 'epoch', 'train', 'validation']
nums_nodes_on_layer_for_csv = []
start_weights_iteration = []
k_folds = []
epochs = []
train_losses = []
validation_losses = []
for num_nodes_layers in nums_nodes_layers:
    num_nodes_str = "_".join([str(el) for el in num_nodes_layers])
    best_validation_loss_model_global = None
    best_validation_loss_global = math.inf
    best_train_val_test_global = None
    for j in range(num_repetitions):
        nn_model = ChassFfnnTanhRegressor(INPUTS, num_nodes_layers, OUTPUTS)
        number_of_parameters = sum(torch.numel(tensor) for tensor in nn_model.state_dict().values())
        models = []
        test_losses = []
        train_losses_end = []
        val_losses_end = []
        nums_epoch = []
        AICs = []
        ti = time.time()
        for i, (train_batches_nums, validation_batches_nums, test_batches_nums) in enumerate(train_validation_test):


            best_validation_loss_model = None
            best_validation_loss = math.inf
            
            train_batches = [Xs_ys[i] for i in train_batches_nums]
            validation_batches = [Xs_ys[i] for i in validation_batches_nums]
            test_batches = [Xs_ys[i] for i in test_batches_nums]
            
            X_train = torch.vstack([batch[0] for batch in train_batches])
            y_train = torch.vstack([batch[1] for batch in train_batches])
            X_validation = torch.vstack([batch[0] for batch in validation_batches])
            y_validation = torch.vstack([batch[1] for batch in validation_batches])
            X_test = torch.vstack([batch[0] for batch in test_batches])
            y_test = torch.vstack([batch[1] for batch in test_batches])
            
            X_train = X_train.to(device='cuda')
            y_train = y_train.to(device='cuda')
            X_validation = X_validation.to(device='cuda')
            y_validation = y_validation.to(device='cuda')
            X_test = X_test.to(device='cuda')
            y_test = y_test.to(device='cuda')
            
            training_data = TensorDataset(X_train, y_train)
            validation_data = TensorDataset(X_validation, y_validation)
            test_data = TensorDataset(X_test, y_test)
            
            train_dataloader = DataLoader(training_data, batch_size=STOCH_REG_BATCH_SIZE, shuffle=True)
            this_iteration_model = ChassFfnnTanhRegressor(INPUTS, num_nodes_layers, OUTPUTS)
            this_iteration_model.load_state_dict(nn_model.state_dict())
            this_iteration_model.to(device='cuda')
            optimizer = optim.Adam(this_iteration_model.parameters(), lr=LR, )
            counter = time_to_wait_for_improvement
            
            for epoch in range(n_epochs):
                this_iteration_model.train()
                for X, y in train_dataloader:
                    y_pred = this_iteration_model(X)
                    loss = loss_fn(y_pred, y)
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                #print(f'Finished epoch {epoch}, latest loss {loss}')
                train_losses.append(float(loss))
                sum_loss = 0
                num_loss = 0
                with torch.no_grad():
                    this_iteration_model.eval()
                    for X, y in validation_data:
                        y_pred = this_iteration_model(X)
                        loss = loss_fn(y_pred, y)
                        sum_loss += loss
                        num_loss += 1
                    mean_loss = float(sum_loss / num_loss)
                if mean_loss < best_validation_loss: #- minimal_improvement:
                    counter = time_to_wait_for_improvement
                    best_validation_loss = mean_loss
                    best_validation_loss_model = ChassFfnnTanhRegressor(INPUTS, num_nodes_layers, OUTPUTS)
                    best_validation_loss_model.to(device='cuda')
                    best_validation_loss_model.load_state_dict(this_iteration_model.state_dict())
                counter -= 1
                if counter < 0:
                    nums_epoch.append(epoch)
                    break
                validation_losses.append(mean_loss)
                #print("validation loss", mean_loss)
                nums_nodes_on_layer_for_csv.append(num_nodes_str)
                start_weights_iteration.append(j)
                k_folds.append(i)
                epochs.append(epoch)
                csv_counter += 1
                if csv_counter == 100:
                    csv_counter = 0
                    df = pandas.DataFrame(zip(nums_nodes_on_layer_for_csv,
                                              start_weights_iteration,
                                              k_folds,
                                              epochs,
                                              train_losses,
                                              validation_losses), columns = columns)
                    if csv_first:
                        csv_first = False
                        df.to_csv(csv_name, index=False)
                    else:
                        df.to_csv(csv_name, mode='a', index=False, header=False)
                    nums_nodes_on_layer_for_csv = []
                    start_weights_iteration = []
                    k_folds = []
                    epochs = []
                    train_losses = []
                    validation_losses = []
            else:
                nums_epoch.append(n_epochs)

            sum_loss = 0
            num_loss = 0
            
            elapsed_time = time.time() - ti
            torch.save(best_validation_loss_model.state_dict(), f"saved_models/Chass_ffnn_tanh_{num_nodes_str}_{j}_{i}")
            
            if best_validation_loss < best_validation_loss_global:
                best_validation_loss_global = best_validation_loss
                best_validation_loss_model_global = best_validation_loss_model
                best_train_val_test_global = [train_batches_nums, validation_batches_nums, test_batches_nums]

            sum_loss = 0
            num_loss = 0
            with torch.no_grad():
                best_validation_loss_model.eval()
                for X, y in test_data:
                    y_pred = best_validation_loss_model(X)
                    loss = loss_fn(y_pred, y)
                    sum_loss += loss
                    num_loss += 1
                mean_test_loss = float(sum_loss / num_loss)

                test_losses.append(mean_test_loss)

                total_train_dataloader = DataLoader(training_data, len(training_data))
                sum_loss = 0
                num_loss = 0
                for X, y in total_train_dataloader:
                    y_pred = best_validation_loss_model(X)
                    Train_Loss = loss_fn(y_pred, y)
                    sum_loss += Train_Loss
                    num_loss += 1
                train_losses_end.append(float(sum_loss/num_loss))
                total_val_dataloader = DataLoader(validation_data, len(validation_data))
                sum_loss = 0
                num_loss = 0
                for X, y in total_val_dataloader:
                    y_pred = best_validation_loss_model(X)
                    Val_Loss = loss_fn(y_pred, y)
                    sum_loss += Val_Loss
                    num_loss += 1
                val_losses_end.append(float(sum_loss/num_loss))
                AICs.append(AICc(number_of_parameters, best_validation_loss_model, n_train, total_train_dataloader))
                

        elapsed_time = time.time() - ti
        mean_test = numpy.mean(test_losses)
        sd_test = numpy.std(test_losses, ddof=1)
        mean_AIC = numpy.mean(AICs)
        sd_AIC = numpy.std(AICs, ddof=1)
        print(mean_test)
        print(sd_test)
        print(nums_epoch)
        with open("Chass_ffnn_tanh.txt", "a") as f:
            f.write(f"dropout {DROPOUT}, number of nodes on layers {num_nodes_str} mean test loss {mean_test}, sd {sd_test}, number of parameters {number_of_parameters}, batch size {STOCH_REG_BATCH_SIZE}, learning rate {LR}\n")
            f.write(f"{test_losses}\n")
            f.write(f"mean AICc: {mean_AIC}, sd AICc: {sd_AIC}\n")
            f.write(f"AICcs: {AICs}\n")
            f.write(f'time {elapsed_time}\n')
            f.write(f"iterations {nums_epoch}\n")
            f.write(f"validation losses {val_losses_end}\n")
            f.write(f"train losses {train_losses_end}\n\n")
    torch.save(best_validation_loss_model_global.state_dict(), f"saved_models/Chass_ffnn_tanh_{num_nodes_str}_best")
    with open(f'saved_models/Chass_ffnn_tanh_{num_nodes_str}_best_train_validation_test.pkl', 'wb') as f:
        pickle.dump(best_train_val_test_global, f)
df = pandas.DataFrame(zip(nums_nodes_on_layer_for_csv,
                          start_weights_iteration,
                          k_folds,
                          epochs,
                          train_losses,
                          validation_losses), columns = columns)
if csv_first:
    csv_first = False
    df.to_csv(csv_name, index=False)
else:
    df.to_csv(csv_name, mode='a', index=False, header=False)

0.009625473711639642
0.0005341465654444253
[63171, 5303, 35857, 8942, 5285, 61314, 149548, 31030, 45447, 36820]
0.009962362423539161
0.0006012855480554464
[36872, 5726, 34428, 65260, 5990, 37333, 6737, 19272, 9519, 19619]
0.00973732341080904
0.0005828894910416948
[56031, 6319, 10777, 48364, 5305, 56276, 9390, 46165, 12121, 46314]
0.008945083152502776
0.0004025112078180471
[33379, 16403, 38096, 31872, 15319, 38330, 28493, 24128, 42969, 24570]
0.009336687065660954
0.0004306010827797243
[10108, 5935, 4539, 30595, 8030, 9633, 10640, 9661, 12350, 10973]
0.009052156656980514
0.0005087760857192834
[9830, 47637, 5441, 20726, 9391, 8825, 12325, 22669, 39023, 22782]
0.008767691347748041
0.0003363983298556955
[20646, 14681, 7073, 17204, 18288, 21373, 16701, 6239, 18534, 6136]
0.008856681920588017
0.0005276292987257043
[14621, 16960, 10157, 7129, 6522, 15255, 3918, 21392, 28127, 18736]
0.009017699770629406
0.0003685245289616729
[14825, 5859, 6414, 7904, 7670, 14407, 10765, 12843, 22331, 11854]
0.0